In [ ]:
import pandas as pd 
from glob import glob 
import sys 
sys.path.append('/home/work/yuna/HPA') 
from preprocessing.utils import MODELNAMES 
root_dir = '/home/work/yuna/HPA/evaluation/scored'

def find_matching(f, targets): 
    for t in targets : 
        if t in f : 
            f = f.replace(f'{t}', '')   
            return t, f 
    print(f"cannot find matching {f} in {targets}") 

def get_summary(dataset='mmstar'): 
    files = glob(f"{root_dir}/*/*{dataset}*.jsonl")  + glob(f"{root_dir}/*/*/*/{dataset}*.jsonl")

    dfs= []
    for f in files: 
        try: 
            df = pd.read_json(f, lines=True)
            if 'finetuned' in f : 
                df['model'] = f.split('/')[-2].replace('fold_0', '')
            else: 
                df['model'], f = find_matching(f, [model.split('/')[-1] for model in MODELNAMES])  
            df['condition'] = f.split('/')[-1][:-6].replace(f'_', ' ').replace('vqa 1k', '').replace(f'{dataset}', '').strip()
            dfs.append(df)
        except Exception as e: 
            print(e)
    df = pd.concat(dfs)
    print(len(files) ) 

    pt = df.pivot_table(
        index=['model'],  
        columns=['condition'], 
        values=['correct'],
        aggfunc=['mean', 'count']
    )
    pt = pt.round(4)
    pt.to_csv(f"./tables/summary_{dataset}.csv")
    return df , pt 

cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-0.6B_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-1.5-7b-hf']
cannot unpack non-iterable NoneType object
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-8B-Base_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-1.5-7b-hf']
cannot unpack non-iterable NoneType object
72
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-4B_spubench.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-

In [54]:
model_results = {}
for ds in ['mmstar', 'spubench', 'vqa_5k', 'vqa_1k']: 
    model_results[ds], pv = get_summary(ds) 

cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-0.6B_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-1.5-7b-hf']
cannot unpack non-iterable NoneType object
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-8B-Base_mmstar.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-v1.6-mistral-7b-hf', 'llava-1.5-7b-hf']
cannot unpack non-iterable NoneType object
72
cannot find matching /home/work/yuna/HPA/evaluation/scored/pretrained/Qwen3-4B_spubench.jsonl in ['InternVL3_5-8B', 'InternVL3_5-4B', 'InternVL3_5-2B', 'InternVL3_5-1B', 'Qwen3-VL-2B-Instruct', 'Qwen3-VL-4B-Instruct', 'Qwen3-VL-8B-Instruct', 'llava-v1.6-vicuna-7b-hf', 'llava-

In [52]:
### VQA Questions 
human_vqa=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_vqa_per_question.csv')
human_vqa['model'] = "humans" 
human_vqa['condition'] = "inst blind" 
human_vqa.rename(columns={"mean_accuracy": 'correct', 'qid': 'question_id'}, inplace=True) 
qids = human_vqa.question_id.unique()
print(len(qids))
human_vqa.columns

374


Index(['Unnamed: 0', 'question_id', 'answer_type', 'num_responses', 'answers',
       'confidences', 'gt_answers', 'visual_gt', 'correct', 'std_accuracy',
       'mean_confidence', 'std_confidence', 'accuracies', 'mean_gt_similarity',
       'std_gt_similarity', 'gt_similarities', 'mean_visual_similarity',
       'std_visual_similarity', 'visual_similarities', 'model', 'condition'],
      dtype='object')

In [53]:
model_vqa = model_results['vqa_1k'] 
model_vqa = model_vqa[model_vqa['question_id'].isin(qids)]
model_vqa.head()

,image_id,question_id,question_type,question,answers,multiple_choice_answer,answer_type,pid,output,correct,model,condition
2,COCO_val2014_000000356421.jpg,356421011,what is the,Question: What is the boy's skateboard balanci...,"[{'answer': 'air', 'answer_confidence': 'no', ...",nothing,other,2,pole,0.0,InternVL3_5-1B,
5,COCO_val2014_000000011241.jpg,11241003,is this,Question: Is this enough food for more than tw...,"[{'answer': 'yes', 'answer_confidence': 'yes',...",yes,yes/no,5,yes,1.0,InternVL3_5-1B,
7,COCO_val2014_000000537701.jpg,537701022,does this,Question: Does this man's tie match the backgr...,"[{'answer': 'no', 'answer_confidence': 'yes', ...",no,yes/no,7,no,1.0,InternVL3_5-1B,
12,COCO_val2014_000000135486.jpg,135486006,what are,Question: What are they carrying? Answer the q...,"[{'answer': 'kite', 'answer_confidence': 'yes'...",kite,other,12,kite,1.0,InternVL3_5-1B,
13,COCO_val2014_000000305329.jpg,305329002,what color is,Question: What color is his helmet? Answer the...,"[{'answer': 'gray and yellow', 'answer_confide...",black,other,13,black,1.0,InternVL3_5-1B,


In [ ]:
vqa_human_comparison = pd.concat([model_vqa[model_vqa['question_id'].isin(qids)] , human_vqa])
# vqa = vqa_human_comparison.groupby(['model', 'condition']).mean(numeric_only=True)['correct'] 
vqa = vqa_human_comparison.pivot_table( 
    index=['model'], 
    columns=['condition'],  # , 'category', 'l2_category' 
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
) 

374


In [ ]:
vqa['blind'] = vqa[('mean', 'correct', 'inst blind')] - vqa[('mean', 'correct', 'blind')]
vqa['MG'] = vqa[('mean', 'correct', '')] - vqa[('mean', 'correct', 'inst blind')] 
vqa.to_csv(f'./tables/vqa_human_comparison.csv')
vqa 

mean            \
                                                    correct             
condition                                                       blind   
model                                                                   
InternVL3_5-1B                                     0.804813  0.435829   
InternVL3_5-2B                                     0.820856  0.455437   
InternVL3_5-4B                                     0.829768  0.450089   
InternVL3_5-8B                                     0.868984  0.471480   
InternVL3_5-8B_A1_vqa_gt                           0.872549       NaN   
InternVL3_5-8B_A2_vqa_10_blind_inst                0.860963       NaN   
InternVL3_5-8B_A3_vqa_15_blind_inst                0.860963       NaN   
InternVL3_5-8B_A4_mmstar_15_blind_inst             0.863636       NaN   
Qwen3-VL-2B-Instruct                               0.829768  0.454545   
Qwen3-VL-4B-Instruct                               0.859180  0.454545   
Qwen3-VL-4B-Instruct_A1_vqa_gt                     0.881462       NaN   
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst          0.844029       NaN   
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst          0.849376       NaN   
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst       0.856506       NaN   
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst         0.847594       NaN   
Qwen3-VL-4B-Instruct_SFT_vqa_gt                    0.886809       NaN   
Qwen3-VL-8B-Instruct                               0.893939  0.442959   
Qwen3-VL-8B-Instruct_A1_vqa_gt                     0.893048       NaN   
Qwen3-VL-8B-Instruct_A2_vqa_10_blind_inst          0.869875       NaN   
Qwen3-VL-8B-Instruct_A3_vqa_15_blind_inst          0.893048       NaN   
Qwen3-VL-8B-Instruct_A4_mmstar_15_blind_inst       0.891266       NaN   
Qwen3-VL-8B-Instruct_SFT_mmstar_15_blind_inst      0.888592       NaN   
Qwen3-VL-8B-Instruct_SFT_vqa_15_blind_inst         0.875223       NaN   
humans                                                  NaN       NaN   
llava-v1.6-mistral-7b-hf                           0.862745  0.450980   
llava-v1.6-mistral-7b-hf_A1_vqa_gt                 0.877005       NaN   
llava-v1.6-mistral-7b-hf_A2_vqa_10_blind_inst      0.836898       NaN   
llava-v1.6-mistral-7b-hf_A4_mmstar_15_blind_inst   0.859180       NaN   
llava-v1.6-mistral-7b-hf_SFT_mmstar_15_blind_inst  0.865419       NaN   

                                                                 blind  \
                                                                         
condition                                         inst blind             
model                                                                    
InternVL3_5-1B                                      0.438503  0.002674   
InternVL3_5-2B                                      0.452763 -0.002674   
InternVL3_5-4B                                      0.472371  0.022282   
InternVL3_5-8B                                      0.453654 -0.017825   
InternVL3_5-8B_A1_vqa_gt                            0.516934       NaN   
InternVL3_5-8B_A2_vqa_10_blind_inst                 0.507130       NaN   
InternVL3_5-8B_A3_vqa_15_blind_inst                 0.495544       NaN   
InternVL3_5-8B_A4_mmstar_15_blind_inst              0.464349       NaN   
Qwen3-VL-2B-Instruct                                0.461676  0.007130   
Qwen3-VL-4B-Instruct                                0.474153  0.019608   
Qwen3-VL-4B-Instruct_A1_vqa_gt                      0.492870       NaN   
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst           0.474153       NaN   
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst           0.484848       NaN   
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst        0.473262       NaN   
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst          0.472371       NaN   
Qwen3-VL-4B-Instruct_SFT_vqa_gt                     0.516934       NaN   
Qwen3-VL-8B-Instruct                                0.474153  0.031194   
Qwen3-VL-8B-Instruct_A1_vqa_gt                      0.461676       NaN   
Qwen3-VL-8B-Instr

In [ ]:
### MMStar questions 
human_mc=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_mc_per_question.csv') 
qids = human_mc.pid.unique() 
model_mc = model_results['mmstar']
human_mc['pid'] = human_mc['pid'].astype('Int64')
model_mc['pid'] = model_mc['pid'].astype('Int64') 
print(len(qids))


In [41]:
pt = model_mc.pivot_table( 
    index=['model', 'pid', 'category', 'l2_category'],  
    columns=['condition'],  
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
)
pt['MG'] = pt[('mean', 'correct', '')] - pt[('mean', 'correct', 'inst blind')] 
pt.to_csv(f'./tables/mmstar_model_MG_by_qid.csv')
pt.pivot_table( 
    index=['model'],  
    columns=['category', 'l2_category'],  
    values=['MG'],
    aggfunc=['mean'] # , 'count' 
).round(3).to_csv(f'./tables/mmstar_model_MG_by_category.csv')

In [ ]:
mmstar_human_comparison = pd.concat([model_mc[model_mc['pid'].isin(qids)] , human_mc])
pt = mmstar_human_comparison.pivot_table( 
    index=['model'], 
    columns=['condition', 'category', 'l2_category'], 
    values=['correct'],
    aggfunc=['mean'] # , 'count' 
)
pt = pt.round(3)
pt.to_csv(f'./tables/mmstar_human_comparison.csv')
# with pd.option_context('display.float_format', '{:0.3f}'.format):
    # display(pt)  
mean_correct = mmstar_human_comparison.groupby(['model', 'condition'])['correct'].count()
mean_correct

model                                              condition 
InternVL3_5-1B                                                   247
                                                   blind         247
                                                   inst blind    247
InternVL3_5-2B                                                   247
                                                   blind         247
                                                                ... 
llava-v1.6-mistral-7b-hf_A4_mmstar_15_blind_inst   inst blind    247
llava-v1.6-mistral-7b-hf_SFT_mmstar_15_blind_inst                247
                                                   inst blind    247
llava-v1.6-mistral-7b-hf_SFT_vqa_15_blind_inst                   247
                                                   inst blind    247
Name: correct, Length: 69, dtype: int64

In [21]:
!python /home/work/yuna/HPA/evaluation/score_humans.py --human_data_dir n20 --with_similarity 

   Session: s1
   Data dir: /home/work/yuna/HPA/data/humans/n20

📚 Loading annotations...
Length of MMStar questions: 267

🔧 Loading sentence transformer...
   ✓ Encoder loaded
Processing VQA (text) responses...
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/dataset/questions/s1.csv
/home/work/yuna/HPA/data/humans/n20/1ed1a464_20251204_110002/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/9d6d8564_20251210_233650/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/74d408e5_20251213_134035/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/99f271ae_20251203_111155/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/ba2d2124_20251208_223639/answers.csv is incomplete, skip
✓ Loaded 12828 responses from 20 files

[2/5] Translating Korean answers...
✓ Loaded 1650 cached translations from /home/work/yuna/HPA/preprocessing/translation_cache.json

🌐 Translation Status:


In [ ]:
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir finetuned  
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir pretrained --with_similarity 

Found 178 files to score
📊 Scoring: /home/work/yuna/HPA/evaluation/results/finetuned/InternVL3_5-8B/InternVL3_5-8B_A3_vqa_15_blind_instfold_0/vqa_1k.jsonl
📂 Loading VQA annotations (this may take a moment)...
   ✓ Loaded 214354 question annotations
   ✓ Saved scored file: /home/work/yuna/HPA/evaluation/scored/finetuned/InternVL3_5-8B/InternVL3_5-8B_A3_vqa_15_blind_instfold_0/vqa_1k.jsonl

📈 Results:
   Accuracy: 0.7500 (750/1000)

   Per-category:
      is the man: 1.000 (9/9)
      does this: 1.000 (14/14)
      is the woman: 1.000 (3/3)
      is there a: 1.000 (21/21)
      is this an: 1.000 (5/5)
      what is the person: 1.000 (2/2)
      what is the color of the: 1.000 (5/5)
      are these: 1.000 (11/11)
      could: 1.000 (7/7)
      is it: 1.000 (15/15)
      has: 1.000 (2/2)
      is that a: 1.000 (4/4)
      was: 1.000 (2/2)
      are there: 1.000 (10/10)
      do you: 1.000 (4/4)
      what sport is: 1.000 (5/5)
      is he: 1.000 (3/3)
      what room is: 1.000 (2/2)
      

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.5,
    'lines.linewidth': 2,
})

# Optional: Color palette
colors = sns.color_palette("colorblind")

In [ ]:
from scipy.stats import wasserstein_distance, ks_2samp

In [ ]:
def plot_vqa_distributions_with_metrics(H, M, model_name): 

    plt.figure(figsize=(6,4))
    sns.histplot(H, bins=20, kde=True, stat="density",
                 alpha=0.5, label="Human avg")
    sns.histplot(M, bins=20, kde=True, stat="density",
                 alpha=0.5, label="Model")

    plt.xlim(0,1)
    plt.xlabel("VQA score")
    plt.title(
        f"{model_name}\n"
        f"Wasserstein={wd:.3f}, KS={ks:.3f} (p={ks_p:.1e})"
    )
    plt.legend()
    plt.tight_layout()
    # plt.show()
    plt.savefig(f"./figures/distribution_{model_name}", dpi=300)  

    wd = wasserstein_distance(H, M)
    ks, ks_p = ks_2samp(H, M)

    return {
        "wasserstein": wd,
        "ks_stat": ks,
        "ks_p": ks_p,
        "human_mean": H.mean(),
        "model_mean": M.mean(),
    }


In [ ]:
models = vqa_human_comparison.model.unique() 
len(models)

28

In [3]:
pt = vqa_human_comparison.pivot_table(
    index=['model'], 
    columns=['condition'], 
    values=['correct'], # , "mean_confidence" 
    aggfunc=['mean', 'count']
)
pt = pt.round(4)
pt.to_csv(f'./tables/vqa_human_comparison.csv')
pt 

NameError: name 'vqa_human_comparison' is not defined

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl